# PySpark DataFrame Interview Exercises

This notebook solves each exercise in a separate Jupyter-style section. Every section contains:

- its own sample data,
- the PySpark solution,
- an explanation of the pattern being tested,
- and interview notes where the correct answer depends on business rules.

The examples use the **PySpark DataFrame API**, not RDDs.


## 0. Notebook Setup

Run this cell first. The remaining sections are independent from one another after the Spark session is created.


In [ ]:
from pathlib import Path
import os
import tempfile

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PySparkInterviewExercises")
    .getOrCreate()
)

# Small value for small local exercises. Production values depend on cluster size.
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

# 1. Data Cleaning

### Tasks

1. Remove duplicate rows.
2. Fill missing ages with the average age.
3. Replace missing names with `"Unknown"`.
4. Filter people whose age is greater than 30.



### 1A. Create the sample DataFrame

In [ ]:
people_schema = T.StructType([
    T.StructField("id", T.IntegerType(), nullable=False),
    T.StructField("name", T.StringType(), nullable=True),
    T.StructField("age", T.DoubleType(), nullable=True),
    T.StructField("city", T.StringType(), nullable=True),
])

people_data = [
    (1, "Alice", 25.0, "NY"),
    (2, "Bob", None, "LA"),
    (2, "Bob", None, "LA"),
    (3, None, 40.0, "NY"),
]

people_df = spark.createDataFrame(people_data, schema=people_schema)
people_df.show()

### 1B. Remove exact duplicate rows

In [ ]:
people_deduplicated_df = people_df.dropDuplicates()
people_deduplicated_df.show()

`dropDuplicates()` compares all columns when no subset is supplied.

To keep one row per ID instead, the code would be:

```python
df.dropDuplicates(["id"])
```

That is a different business rule because two rows can share an ID while containing different values.

### 1C. Calculate the average non-null age

In [ ]:
average_age = (
    people_deduplicated_df
    .agg(F.avg("age").alias("average_age"))
    .first()["average_age"]
)

print("Average age:", average_age)

### 1D. Fill the missing values

In [ ]:
people_clean_df = people_deduplicated_df.fillna({
    "age": average_age,
    "name": "Unknown",
})

people_clean_df.show()

### 1E. Filter people over 30

In [ ]:
people_over_30_df = people_clean_df.filter(F.col("age") > 30)
people_over_30_df.show()

### Interview explanation

- `dropDuplicates()` removes repeated records.
- `avg()` ignores null values by default.
- `fillna()` replaces nulls using either one value or a column-to-value dictionary.
- `filter()` keeps rows that satisfy a Boolean condition.
- The data type matters. The age column was defined as `DoubleType` so the average `32.5` is not truncated to an integer.

# 2. Group By and Aggregation

### Questions

1. What is the total spend per customer?
2. What is the average purchase amount?
3. Who is the highest-spending customer?

### 2A. Create the sales DataFrame

In [ ]:
sales_data = [
    ("A", "Phone", 500.0),
    ("A", "Laptop", 1000.0),
    ("B", "Phone", 700.0),
]

sales_df = spark.createDataFrame(
    sales_data,
    ["customer", "product", "amount"]
)

sales_df.show()

### 2B. Total spend per customer

In [ ]:
customer_spend_df = (
    sales_df
    .groupBy("customer")
    .agg(F.sum("amount").alias("total_spend"))
    .orderBy("customer")
)

customer_spend_df.show()

### 2C. Overall average purchase amount

In [ ]:
overall_average_purchase_df = sales_df.agg(
    F.avg("amount").alias("average_purchase_amount")
)

overall_average_purchase_df.show()

### 2D. Average purchase amount per customer

In [ ]:
customer_average_purchase_df = (
    sales_df
    .groupBy("customer")
    .agg(F.avg("amount").alias("average_purchase_amount"))
    .orderBy("customer")
)

customer_average_purchase_df.show()

### 2E. Highest-spending customer

In [ ]:
highest_spending_customer_df = (
    customer_spend_df
    .orderBy(F.desc("total_spend"), F.asc("customer"))
    .limit(1)
)

highest_spending_customer_df.show()

### Interview explanation

`groupBy()` defines the level at which rows are combined. `agg()` performs one or more calculations for each group.

A useful mental pattern is:

```python
(
    df
    .groupBy(grouping_columns)
    .agg(aggregate_expressions)
    .orderBy(sort_expressions)
)
```

Always clarify whether “average purchase amount” means the average across the entire dataset or the average for each customer.

# 3. Top N per Group

### Goal

Return the highest-paid employee in each department.

A window function is needed because we want to rank rows **inside each department** without collapsing the rows as `groupBy()` would.

### 3A. Create the employee DataFrame

In [ ]:
employee_data = [
    ("IT", "John", 100),
    ("IT", "Mary", 200),
    ("IT", "Alex", 150),
    ("HR", "Sara", 180),
    ("HR", "Mike", 180),
    ("HR", "Nina", 140),
]

employees_df = spark.createDataFrame(
    employee_data,
    ["department", "employee", "salary"]
)

employees_df.show()

### 3B. Highest-paid employee using `row_number()`

In [ ]:
department_salary_window = (
    Window
    .partitionBy("department")
    .orderBy(F.desc("salary"), F.asc("employee"))
)

highest_paid_row_number_df = (
    employees_df
    .withColumn(
        "row_number",
        F.row_number().over(department_salary_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .orderBy("department")
)

highest_paid_row_number_df.show()

`row_number()` always assigns exactly one row the number `1`. The secondary sort by employee name makes a salary tie deterministic.

### 3C. Compare `row_number`, `rank`, and `dense_rank`

In [ ]:
rank_comparison_window = (
    Window
    .partitionBy("department")
    .orderBy(F.desc("salary"))
)

rank_comparison_df = (
    employees_df
    .withColumn("row_number", F.row_number().over(rank_comparison_window))
    .withColumn("rank", F.rank().over(rank_comparison_window))
    .withColumn("dense_rank", F.dense_rank().over(rank_comparison_window))
    .orderBy("department", F.desc("salary"), "employee")
)

rank_comparison_df.show()

### 3D. Return every employee tied for highest salary

In [ ]:
highest_paid_with_ties_df = (
    employees_df
    .withColumn("salary_rank", F.dense_rank().over(rank_comparison_window))
    .filter(F.col("salary_rank") == 1)
    .drop("salary_rank")
    .orderBy("department", "employee")
)

highest_paid_with_ties_df.show()

### Rank differences

- `row_number()`: always produces `1, 2, 3, ...`; ties still receive different numbers.
- `rank()`: tied rows share a rank, and the next rank contains a gap. Example: `1, 1, 3`.
- `dense_rank()`: tied rows share a rank, with no gap. Example: `1, 1, 2`.

Use `row_number()` when exactly one row must be chosen. Use `rank()` or `dense_rank()` when ties should be retained.

# 4. Joins

### Questions

1. Perform an inner join.
2. Perform a left join.
3. Find customers with no orders.
4. Calculate total spend per customer.

### 4A. Create customers and orders

In [ ]:
customers_data = [
    (1, "Alice"),
    (2, "Bob"),
]

orders_data = [
    (1, 100.0),
    (1, 50.0),
    (3, 25.0),
]

customers_df = spark.createDataFrame(customers_data, ["id", "name"])
orders_df = spark.createDataFrame(orders_data, ["customer_id", "amount"])

print("Customers")
customers_df.show()

print("Orders")
orders_df.show()

### 4B. Inner join

In [ ]:
inner_join_df = customers_df.join(
    orders_df,
    customers_df.id == orders_df.customer_id,
    "inner"
)

inner_join_df.show()

An inner join keeps only matching records. Order `customer_id = 3` disappears because customer 3 is not present, and Bob disappears because he has no order.

### 4C. Left join

In [ ]:
left_join_df = customers_df.join(
    orders_df,
    customers_df.id == orders_df.customer_id,
    "left"
)

left_join_df.show()

A left join keeps every row from `customers_df`. Bob remains, but the order columns are null.

### 4D. Customers with no orders — preferred `left_anti` solution

In [ ]:
customers_with_no_orders_df = customers_df.join(
    orders_df,
    customers_df.id == orders_df.customer_id,
    "left_anti"
)

customers_with_no_orders_df.show()

A `left_anti` join returns rows from the left side that have **no match** on the right side. This directly expresses the business question.

### 4E. Customers with no orders — left join plus null filter

In [ ]:
customers_with_no_orders_alternative_df = (
    customers_df
    .join(
        orders_df,
        customers_df.id == orders_df.customer_id,
        "left"
    )
    .filter(F.col("customer_id").isNull())
    .select("id", "name")
)

customers_with_no_orders_alternative_df.show()

### 4F. Total spend per customer, including customers with zero orders

In [ ]:
total_spend_per_customer_df = (
    customers_df
    .join(
        orders_df,
        customers_df.id == orders_df.customer_id,
        "left"
    )
    .groupBy("id", "name")
    .agg(F.sum("amount").alias("total_spend"))
    .fillna({"total_spend": 0.0})
    .orderBy("id")
)

total_spend_per_customer_df.show()

### Interview explanation

- Inner join: only matched rows.
- Left join: every left-side row, with nulls for missing matches.
- Left anti join: left-side rows with no right-side match.
- A left join is required before aggregation when customers with zero orders must remain in the result.

# 5. Find Duplicate Records

### Tasks

1. Find duplicate emails.
2. Count each email's occurrences.
3. Keep one row per email.
4. Keep only emails that originally occurred exactly once.

The last two interpretations are often confused, so both are shown.

### 5A. Create the email DataFrame

In [ ]:
email_data = [
    ("a@test.com",),
    ("b@test.com",),
    ("a@test.com",),
    ("c@test.com",),
]

emails_df = spark.createDataFrame(email_data, ["email"])
emails_df.show()

### 5B. Count occurrences

In [ ]:
email_counts_df = (
    emails_df
    .groupBy("email")
    .agg(F.count(F.lit(1)).alias("occurrence_count"))
    .orderBy("email")
)

email_counts_df.show()

### 5C. Find duplicate emails

In [ ]:
duplicate_emails_df = email_counts_df.filter(
    F.col("occurrence_count") > 1
)

duplicate_emails_df.show()

### 5D. Keep one copy of every email

In [ ]:
one_row_per_email_df = emails_df.dropDuplicates(["email"]).orderBy("email")
one_row_per_email_df.show()

### 5E. Keep only emails that appeared exactly once

In [ ]:
originally_unique_emails_df = (
    email_counts_df
    .filter(F.col("occurrence_count") == 1)
    .select("email")
    .orderBy("email")
)

originally_unique_emails_df.show()

### Interview distinction

- `dropDuplicates(["email"])` keeps one representative row for every email, including `a@test.com`.
- Filtering `count == 1` removes every value that was duplicated, so `a@test.com` is excluded completely.

# 6. Window Function Challenge

### Questions

1. Running total.
2. Previous transaction amount.
3. Difference from the previous transaction.
4. Rolling seven-day average.

### 6A. Create transactions with real dates

In [ ]:
transaction_data = [
    ("A", "2026-01-01", 10.0),
    ("A", "2026-01-02", 20.0),
    ("A", "2026-01-03", 15.0),
    ("A", "2026-01-09", 40.0),
    ("B", "2026-01-01", 5.0),
    ("B", "2026-01-05", 25.0),
]

transactions_df = (
    spark.createDataFrame(transaction_data, ["user", "date", "amount"])
    .withColumn("date", F.to_date("date"))
)

transactions_df.orderBy("user", "date").show()

### 6B. Running total, previous amount, and difference

In [ ]:
ordered_user_window = (
    Window
    .partitionBy("user")
    .orderBy("date")
)

cumulative_user_window = ordered_user_window.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

transaction_metrics_df = (
    transactions_df
    .withColumn(
        "running_total",
        F.sum("amount").over(cumulative_user_window)
    )
    .withColumn(
        "previous_amount",
        F.lag("amount", 1).over(ordered_user_window)
    )
    .withColumn(
        "difference_from_previous",
        F.col("amount") - F.col("previous_amount")
    )
    .orderBy("user", "date")
)

transaction_metrics_df.show()

### 6C. Optional: next transaction using `lead()`

In [ ]:
transactions_with_next_df = (
    transactions_df
    .withColumn(
        "next_amount",
        F.lead("amount", 1).over(ordered_user_window)
    )
    .orderBy("user", "date")
)

transactions_with_next_df.show()

### 6D. Rolling seven-day average

In [ ]:
# rangeBetween requires a numeric ordering column.
# Unix seconds let us define a true time-based seven-day window.
transactions_with_seconds_df = transactions_df.withColumn(
    "event_seconds",
    F.col("date").cast("timestamp").cast("long")
)

seven_day_seconds = 6 * 24 * 60 * 60

rolling_seven_day_window = (
    Window
    .partitionBy("user")
    .orderBy("event_seconds")
    .rangeBetween(-seven_day_seconds, 0)
)

rolling_average_df = (
    transactions_with_seconds_df
    .withColumn(
        "rolling_7_day_average",
        F.avg("amount").over(rolling_seven_day_window)
    )
    .drop("event_seconds")
    .orderBy("user", "date")
)

rolling_average_df.show()

### Why use `rangeBetween()` for seven days?

`rowsBetween(-6, 0)` means “the current row plus the previous six rows.” It does **not** mean the previous seven calendar days.

A time-based rolling calculation should order by a numeric timestamp and use `rangeBetween()`.

# 7. Sessionization

### Rule

A new session begins when the time gap from the previous event is greater than 30 minutes.

### Pattern

1. Sort events for each user.
2. Use `lag()` to retrieve the previous timestamp.
3. Calculate the gap.
4. Mark session starts with `1`.
5. Cumulatively sum the markers to produce a session number.

### 7A. Create clickstream events

In [ ]:
clickstream_data = [
    ("A", "2026-01-01 09:00:00"),
    ("A", "2026-01-01 09:10:00"),
    ("A", "2026-01-01 09:45:00"),  # 35-minute gap: new session
    ("A", "2026-01-01 10:00:00"),
    ("B", "2026-01-01 12:00:00"),
    ("B", "2026-01-01 12:30:00"),  # exactly 30 minutes: same session
    ("B", "2026-01-01 13:01:00"),  # 31-minute gap: new session
]

clickstream_df = (
    spark.createDataFrame(clickstream_data, ["user", "event_timestamp"])
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
)

clickstream_df.orderBy("user", "event_timestamp").show(truncate=False)

### 7B. Build session IDs

In [ ]:
event_order_window = (
    Window
    .partitionBy("user")
    .orderBy("event_timestamp")
)

session_cumulative_window = event_order_window.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

sessionized_df = (
    clickstream_df
    .withColumn(
        "previous_timestamp",
        F.lag("event_timestamp").over(event_order_window)
    )
    .withColumn(
        "gap_minutes",
        (
            F.col("event_timestamp").cast("long")
            - F.col("previous_timestamp").cast("long")
        ) / 60.0
    )
    .withColumn(
        "is_new_session",
        F.when(
            F.col("previous_timestamp").isNull()
            | (F.col("gap_minutes") > 30),
            1
        ).otherwise(0)
    )
    .withColumn(
        "session_number",
        F.sum("is_new_session").over(session_cumulative_window)
    )
    .withColumn(
        "session_id",
        F.concat_ws(
            "_",
            F.col("user"),
            F.lpad(F.col("session_number").cast("string"), 3, "0")
        )
    )
    .orderBy("user", "event_timestamp")
)

sessionized_df.show(truncate=False)

### Boundary rule

The requirement says the gap must **exceed** 30 minutes. Therefore:

- 30 minutes → same session
- 30 minutes and 1 second → new session

Using `>= 30` would implement a different rule.

# 8. Pivot Challenge

Convert product values into columns.

### 8A. Create the monthly sales DataFrame

In [ ]:
monthly_sales_data = [
    ("Jan", "A", 10),
    ("Jan", "B", 15),
    ("Feb", "A", 12),
]

monthly_sales_df = spark.createDataFrame(
    monthly_sales_data,
    ["Month", "Product", "Sales"]
)

monthly_sales_df.show()

### 8B. Pivot products into columns

In [ ]:
pivoted_sales_df = (
    monthly_sales_df
    .groupBy("Month")
    .pivot("Product", ["A", "B"])
    .agg(F.sum("Sales"))
    .orderBy(
        F.when(F.col("Month") == "Jan", 1)
         .when(F.col("Month") == "Feb", 2)
         .otherwise(99)
    )
)

pivoted_sales_df.show()

Providing the expected pivot values `['A', 'B']` avoids a separate Spark job that discovers all distinct product values. This can matter when the product column has high cardinality.

# 9. Explode Nested Data

Convert each array element into its own row.

### 9A. Create the array DataFrame

In [ ]:
student_subject_data = [
    ("Alice", ["Math", "Science"]),
    ("Bob", ["History"]),
]

student_subjects_df = spark.createDataFrame(
    student_subject_data,
    ["name", "subjects"]
)

student_subjects_df.show(truncate=False)
student_subjects_df.printSchema()

### 9B. Explode the array

In [ ]:
exploded_subjects_df = (
    student_subjects_df
    .select(
        "name",
        F.explode("subjects").alias("subject")
    )
    .orderBy("name", "subject")
)

exploded_subjects_df.show()

Useful variations:

- `explode()` drops rows whose array is null or empty.
- `explode_outer()` preserves a row and returns null when the array is null or empty.
- `posexplode()` also returns each element's array position.

# 10. JSON Parsing

### Tasks

1. Parse a JSON string.
2. Apply an explicit schema.
3. Flatten the nested fields.

### 10A. Create raw JSON data

In [ ]:
raw_json_data = [
    ('{"customer":{"id":1,"city":"NY"}}',),
    ('{"customer":{"id":2,"city":"LA"}}',),
]

raw_json_df = spark.createDataFrame(raw_json_data, ["json_text"])
raw_json_df.show(truncate=False)

### 10B. Define the JSON schema

In [ ]:
customer_json_schema = T.StructType([
    T.StructField(
        "customer",
        T.StructType([
            T.StructField("id", T.IntegerType(), nullable=True),
            T.StructField("city", T.StringType(), nullable=True),
        ]),
        nullable=True,
    )
])

print(customer_json_schema.simpleString())

### 10C. Parse and inspect the nested struct

In [ ]:
parsed_json_df = raw_json_df.withColumn(
    "parsed",
    F.from_json("json_text", customer_json_schema)
)

parsed_json_df.select("json_text", "parsed").show(truncate=False)
parsed_json_df.printSchema()

### 10D. Flatten the nested fields

In [ ]:
flattened_customer_df = parsed_json_df.select(
    F.col("parsed.customer.id").alias("customer_id"),
    F.col("parsed.customer.city").alias("customer_city"),
)

flattened_customer_df.show()

### Interview explanation

- `from_json()` converts a string into a typed Spark struct.
- An explicit schema avoids repeated schema inference and makes failures predictable.
- Nested fields are selected with dot notation such as `parsed.customer.id`.
- Flattening does not alter the original JSON; it projects nested values into top-level columns.

# 11. Skew Handling

### Scenario

One customer has 50 million transactions while most customers have only a few hundred.

### Why the job becomes slow

Spark hashes the grouping or joining key to a partition. All rows for the unusually large customer can land in one partition. Most tasks finish quickly, but one task processes an enormous amount of data and becomes the **straggler**.

Possible symptoms include:

- one task taking much longer than all others,
- large shuffle read or write for one partition,
- memory spilling to disk,
- executor out-of-memory failures,
- poor CPU utilization near the end of a stage.

### 11A. Create a tiny demonstration of a hot key

In [ ]:
skewed_transactions_data = (
    [("HOT_CUSTOMER", float(i % 10)) for i in range(1000)]
    + [("CUSTOMER_A", 10.0), ("CUSTOMER_B", 20.0), ("CUSTOMER_C", 30.0)]
)

skewed_transactions_df = spark.createDataFrame(
    skewed_transactions_data,
    ["customer_id", "amount"]
)

(
    skewed_transactions_df
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"))
    .show()
)

### 11B. Salt a skewed aggregation

In [ ]:
number_of_salts = 8

salted_transactions_df = skewed_transactions_df.withColumn(
    "salt",
    (F.rand(seed=42) * number_of_salts).cast("int")
)

partial_totals_df = (
    salted_transactions_df
    .groupBy("customer_id", "salt")
    .agg(F.sum("amount").alias("partial_total"))
)

salted_final_totals_df = (
    partial_totals_df
    .groupBy("customer_id")
    .agg(F.sum("partial_total").alias("total_amount"))
    .orderBy(F.desc("total_amount"))
)

salted_final_totals_df.show()

Salting splits one hot key into several temporary keys. The first aggregation can run across multiple partitions, and the second aggregation recombines the partial results.

### 11C. Broadcast a small dimension table

In [ ]:
customer_dimension_df = spark.createDataFrame(
    [
        ("HOT_CUSTOMER", "Enterprise"),
        ("CUSTOMER_A", "Retail"),
        ("CUSTOMER_B", "Retail"),
        ("CUSTOMER_C", "SMB"),
    ],
    ["customer_id", "segment"]
)

broadcast_join_df = skewed_transactions_df.join(
    F.broadcast(customer_dimension_df),
    "customer_id",
    "left"
)

broadcast_join_df.show(5)

### 11D. Enable Adaptive Query Execution skew handling

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print(
    "AQE skew join enabled:",
    spark.conf.get("spark.sql.adaptive.skewJoin.enabled")
)

### Which technique solves which problem?

- **Salting keys:** useful when a small number of keys dominate an aggregation or shuffle join.
- **Broadcast join:** useful when one side of the join is small enough to copy to every executor.
- **Repartitioning:** useful when the general partition count or distribution is poor, but `repartition("customer_id")` alone does not split one hot customer across partitions.
- **AQE skew join optimization:** Spark can detect unusually large shuffled join partitions and split them at runtime.
- **Filtering or pre-aggregation:** reduce data before the expensive join whenever the business logic allows it.

A strong interview answer explains that no single method is automatically correct; the fix depends on whether the skew occurs during a join, aggregation, or both.

# 13. Optimize a Slow Job

Original pattern:

```python
df.join(df2, "id").groupBy("city").count()
```

The original code is not automatically wrong. Optimization depends on table sizes, filters, file format, partitioning, and whether either DataFrame is reused.

### 13A. Create demonstration DataFrames

In [ ]:
main_data = [
    (1, "NY", "active"),
    (2, "LA", "inactive"),
    (3, "NY", "active"),
    (4, "Chicago", "active"),
]

lookup_data = [
    (1, "gold"),
    (3, "silver"),
    (4, "gold"),
]

main_df = spark.createDataFrame(main_data, ["id", "city", "status"])
lookup_df = spark.createDataFrame(lookup_data, ["id", "tier"])

### 13B. Baseline solution

In [ ]:
baseline_result_df = (
    main_df
    .join(lookup_df, "id", "inner")
    .groupBy("city")
    .count()
)

baseline_result_df.show()

### 13C. A more selective and broadcast-friendly version

In [ ]:
optimized_main_df = (
    main_df
    .filter(F.col("status") == "active")
    .select("id", "city")
)

optimized_lookup_df = lookup_df.select("id")

optimized_result_df = (
    optimized_main_df
    .join(F.broadcast(optimized_lookup_df), "id", "inner")
    .groupBy("city")
    .count()
)

optimized_result_df.show()

### 13D. Inspect the physical execution plan

In [ ]:
optimized_result_df.explain(mode="formatted")

### Optimization checklist

1. **Filter early:** reduce rows before joining or grouping.
2. **Select only needed columns:** reduce data serialization and shuffle width.
3. **Broadcast only a genuinely small table:** confirm its size instead of broadcasting blindly.
4. **Use partition pruning:** filter partition columns when reading partitioned data.
5. **Use predicate pushdown:** Parquet and ORC can skip row groups based on supported filters.
6. **Avoid unnecessary shuffles:** joins, `groupBy`, `distinct`, `orderBy`, and `repartition` commonly shuffle data.
7. **Cache only when reused:** caching a DataFrame used once adds overhead rather than removing it.
8. **Choose a reasonable shuffle partition count:** too few creates large tasks; too many creates scheduler and tiny-task overhead.
9. **Inspect the Spark UI and `explain()`:** optimize the measured bottleneck, not a guessed bottleneck.

# 14. Read Multiple Files

### Tasks

1. Read every CSV in a directory.
2. Use an explicit schema.
3. Add the source filename.
4. Capture malformed records.

This section creates temporary CSV files so the example can run locally.

### 14A. Create temporary CSV files

In [ ]:
input_directory = Path(tempfile.mkdtemp(prefix="pyspark_csv_input_"))

(input_directory / "part_1.csv").write_text(
    "id,name,age\n"
    "1,Alice,25\n"
    "2,Bob,30\n",
    encoding="utf-8"
)

(input_directory / "part_2.csv").write_text(
    "id,name,age\n"
    "3,Charlie,35\n"
    "4,Diana,not_a_number\n"
    "5,Extra,40,unexpected_field\n",
    encoding="utf-8"
)

print("Input directory:", input_directory)
print("Files:", [path.name for path in input_directory.iterdir()])

### 14B. Define the schema and read all CSV files

In [ ]:
csv_schema = T.StructType([
    T.StructField("id", T.IntegerType(), nullable=True),
    T.StructField("name", T.StringType(), nullable=True),
    T.StructField("age", T.IntegerType(), nullable=True),
    T.StructField("_corrupt_record", T.StringType(), nullable=True),
])

all_csv_files_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(csv_schema)
    .csv(str(input_directory / "*.csv"))
    .withColumn("source_filename", F.input_file_name())
)

all_csv_files_df.show(truncate=False)
all_csv_files_df.printSchema()

### 14C. Separate usable and malformed records

In [ ]:
malformed_records_df = all_csv_files_df.filter(
    F.col("_corrupt_record").isNotNull()
)

usable_records_df = (
    all_csv_files_df
    .filter(F.col("_corrupt_record").isNull())
    .drop("_corrupt_record")
)

print("Usable records")
usable_records_df.show(truncate=False)

print("Malformed records")
malformed_records_df.show(truncate=False)

### Production notes

- `spark.read.csv("/path/*.csv")` or `spark.read.csv("/path/")` reads multiple files.
- `inferSchema=True` is convenient for exploration, but an explicit schema is safer and faster for repeatable pipelines.
- `input_file_name()` records the source path for lineage and debugging.
- `PERMISSIVE` attempts to keep malformed data; `DROPMALFORMED` drops it; `FAILFAST` stops immediately.
- A type-conversion failure can appear as a null even when the complete line is not placed in `_corrupt_record`. Production validation should therefore check required fields and data types in addition to the corrupt-record column.

# 15. End-to-End ETL Challenge

### Requirements

- Clean missing values.
- Remove duplicates.
- Join orders to customers.
- Calculate total sales by city.
- Calculate average order value by segment.
- Find the top three customers by spending.
- Write an output partitioned by city.

### Explicit cleaning assumptions

Interview problems rarely define every business rule. This solution states its assumptions:

1. `order_id`, `customer_id`, and a valid order date are required.
2. A missing order amount is filled with `0.0` for this exercise. In a financial production system, it may be better to quarantine the record instead.
3. Duplicate orders are identified by `order_id`.
4. Duplicate customers are identified by `customer_id`.
5. Orders without a matching customer are preserved and labeled `Unknown`.

### 15A. Create raw orders and customers

In [ ]:
orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), nullable=True),
    T.StructField("customer_id", T.IntegerType(), nullable=True),
    T.StructField("date", T.StringType(), nullable=True),
    T.StructField("amount", T.DoubleType(), nullable=True),
])

raw_orders_data = [
    (1001, 1, "2026-01-01", 100.0),
    (1001, 1, "2026-01-01", 100.0),  # duplicate order
    (1002, 1, "2026-01-02", 50.0),
    (1003, 2, "2026-01-03", 200.0),
    (1004, 3, None, 150.0),            # missing required date
    (1005, 4, "2026-01-04", None),    # missing amount
    (1006, None, "2026-01-05", 90.0), # missing customer ID
    (1007, 5, "2026-01-06", 300.0),   # no matching customer dimension row
    (1008, 2, "2026-01-07", 75.0),
    (1009, 4, "2026-01-08", 125.0),
]

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), nullable=True),
    T.StructField("city", T.StringType(), nullable=True),
    T.StructField("segment", T.StringType(), nullable=True),
])

raw_customers_data = [
    (1, "NY", "Retail"),
    (1, "NY", "Retail"),  # duplicate customer
    (2, "LA", "Enterprise"),
    (3, "NY", "Retail"),
    (4, None, "SMB"),      # missing city
]

raw_orders_df = spark.createDataFrame(raw_orders_data, schema=orders_schema)
raw_customers_df = spark.createDataFrame(raw_customers_data, schema=customers_schema)

print("Raw orders")
raw_orders_df.show()

print("Raw customers")
raw_customers_df.show()

### 15B. Clean the orders

In [ ]:
clean_orders_df = (
    raw_orders_df
    .dropDuplicates(["order_id"])
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("date").isNotNull()
    )
    .withColumn("date", F.to_date("date"))
    .filter(F.col("date").isNotNull())
    .fillna({"amount": 0.0})
)

clean_orders_df.orderBy("order_id").show()

### 15C. Clean the customers

In [ ]:
clean_customers_df = (
    raw_customers_df
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .fillna({
        "city": "Unknown",
        "segment": "Unknown",
    })
)

clean_customers_df.orderBy("customer_id").show()

### 15D. Join the datasets and preserve unmatched orders

In [ ]:
enriched_orders_df = (
    clean_orders_df
    .join(clean_customers_df, "customer_id", "left")
    .fillna({
        "city": "Unknown",
        "segment": "Unknown",
    })
)

enriched_orders_df.orderBy("order_id").show()

### 15E. Total sales by city

In [ ]:
total_sales_by_city_df = (
    enriched_orders_df
    .groupBy("city")
    .agg(F.sum("amount").alias("total_sales"))
    .orderBy(F.desc("total_sales"), "city")
)

total_sales_by_city_df.show()

### 15F. Average order value by segment

In [ ]:
average_order_value_by_segment_df = (
    enriched_orders_df
    .groupBy("segment")
    .agg(F.avg("amount").alias("average_order_value"))
    .orderBy("segment")
)

average_order_value_by_segment_df.show()

### 15G. Top three customers by total spending

In [ ]:
customer_totals_df = (
    enriched_orders_df
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total_spend"))
)

top_three_customers_df = (
    customer_totals_df
    .orderBy(F.desc("total_spend"), F.asc("customer_id"))
    .limit(3)
)

top_three_customers_df.show()

For a global top three, `orderBy(...).limit(3)` is simpler than a global window. A window is especially useful for **top N per city, department, segment, or another group**.

### 15H. Bonus: top three customers within each city using a window

In [ ]:
customer_city_totals_df = (
    enriched_orders_df
    .groupBy("city", "customer_id")
    .agg(F.sum("amount").alias("total_spend"))
)

city_spending_window = (
    Window
    .partitionBy("city")
    .orderBy(F.desc("total_spend"), F.asc("customer_id"))
)

top_three_customers_per_city_df = (
    customer_city_totals_df
    .withColumn(
        "spending_position",
        F.row_number().over(city_spending_window)
    )
    .filter(F.col("spending_position") <= 3)
    .orderBy("city", "spending_position")
)

top_three_customers_per_city_df.show()

### 15I. Write enriched orders partitioned by city

In [ ]:
etl_output_root = Path(tempfile.mkdtemp(prefix="pyspark_etl_output_"))
partitioned_output_path = etl_output_root / "orders_by_city"

(
    enriched_orders_df
    .write
    .mode("overwrite")
    .partitionBy("city")
    .parquet(str(partitioned_output_path))
)

print("Partitioned output:", partitioned_output_path)
print("Created entries:")
for path in sorted(partitioned_output_path.iterdir()):
    print(" -", path.name)

### 15J. Read the partitioned result back

In [ ]:
written_orders_df = spark.read.parquet(str(partitioned_output_path))
written_orders_df.orderBy("order_id").show()
written_orders_df.printSchema()

### ETL interview checklist

A strong end-to-end answer usually explains:

1. which columns are required,
2. what defines a duplicate,
3. how null values are handled,
4. whether unmatched joins should be preserved,
5. the grain of each aggregate,
6. how ties are handled in a top-N calculation,
7. why the chosen output partition column is useful,
8. and how data quality failures would be monitored or quarantined.

# Common Follow-Up Questions

## What causes a shuffle, and why is it expensive?

A shuffle occurs when Spark must redistribute records across partitions so related keys or globally ordered rows can be processed together. Common causes include:

- `groupBy` and `groupByKey`,
- joins that are not broadcast joins,
- `distinct` and `dropDuplicates`,
- `orderBy`,
- window functions with partitioning or ordering,
- `repartition`.

A shuffle is expensive because it can require serialization, network transfer, disk spill, sorting, and synchronization between stages.

---

## When would you use `cache()` versus `persist()`?

`cache()` is shorthand for Spark's default persistence level. `persist()` lets you choose a storage level, such as memory-only or memory-and-disk.

Use them when the same expensive DataFrame is reused by multiple downstream actions. Do not cache a DataFrame used only once. Materialize the cache with an action and call `unpersist()` when it is no longer needed.

---

## How do narrow and wide transformations differ?

A narrow transformation lets each output partition obtain its records from a small number of input partitions, often one. Examples include `select`, `filter`, and many `withColumn` operations.

A wide transformation requires records to move across partitions. Examples include `groupBy`, many joins, `distinct`, and `repartition`. Wide transformations normally create shuffle boundaries and new stages.

---

## When should you use a broadcast join?

Use a broadcast join when one side is small enough to fit safely in each executor's memory. Broadcasting avoids shuffling the large side of the join.

Do not broadcast blindly. Consider the table's serialized size, executor memory, concurrent workloads, and whether statistics are reliable.

---

## How do `coalesce()` and `repartition()` differ?

- `repartition(n)` can increase or decrease partitions and normally performs a full shuffle. It is useful for redistributing data more evenly.
- `coalesce(n)` is mainly used to reduce partitions with less data movement. It can create uneven partitions because it does not fully rebalance the data.

Use `coalesce()` for a modest reduction after filtering. Use `repartition()` when balanced distribution matters.

---

## How do you diagnose a slow Spark job?

1. Identify the slow job, stage, and task in the Spark UI.
2. Compare task durations and shuffle sizes to find skew or stragglers.
3. Check spill, garbage collection, input size, and executor failures.
4. Inspect the physical plan with `explain("formatted")`.
5. Check whether filters were pushed down and partitions were pruned.
6. Check join strategies, partition counts, and data skew.
7. Change one bottleneck at a time and measure again.

---

## What are jobs, stages, tasks, and partitions?

- **Job:** created when an action such as `show`, `count`, `collect`, or `write` is called.
- **Stage:** a group of tasks that can run without crossing a shuffle boundary.
- **Task:** the unit of work applied to one partition during a stage.
- **Partition:** a logical chunk of distributed data.

A stage usually contains approximately one task per input partition.

---

## How does lazy evaluation affect execution?

Transformations such as `filter`, `select`, and `join` build a logical plan but do not immediately process the data. An action triggers optimization and execution.

Lazy evaluation lets Spark combine operations, push filters closer to the data source, remove unnecessary columns, and choose physical strategies. It also means an error in a transformation may not appear until an action is executed.